In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import ast
import copy
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [3]:
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
drive_root = "/content/drive/MyDrive"

pos_matches = []

for root, dirs, files in os.walk(drive_root):
    needed = {
        "scenario23_pos_beam_train.csv",
        "scenario23_pos_beam_val.csv",
        "scenario23_pos_beam_test.csv"
    }
    if needed.issubset(set(files)):
        pos_matches.append(root)

print("Position CSV folder candidates:")
for p in pos_matches:
    print(p)

Position CSV folder candidates:
/content/drive/MyDrive/Pos beam


In [5]:
POS_ROOT = "/content/drive/MyDrive/Pos beam"

pos_train_csv = os.path.join(POS_ROOT, "scenario23_pos_beam_train.csv")
pos_val_csv   = os.path.join(POS_ROOT, "scenario23_pos_beam_val.csv")
pos_test_csv  = os.path.join(POS_ROOT, "scenario23_pos_beam_test.csv")

print(os.path.exists(pos_train_csv), pos_train_csv)
print(os.path.exists(pos_val_csv), pos_val_csv)
print(os.path.exists(pos_test_csv), pos_test_csv)

True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_train.csv
True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_val.csv
True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_test.csv


In [6]:
train_df = pd.read_csv(pos_train_csv)
val_df   = pd.read_csv(pos_val_csv)
test_df  = pd.read_csv(pos_test_csv)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

display(train_df.head())
print(train_df.columns.tolist())

Train: (6832, 3)
Val  : (3416, 3)
Test : (1139, 3)


,index,unit2_pos,unit1_beam
0,3532,"[0.8092883966431671, 0.521083920903955]",17
1,2224,"[0.4816276084988933, 0.29434536152734486]",14
2,9416,"[0.220278556834608, 0.4136596156292844]",17
3,8510,"[0.21412273613497904, 0.4547214157104936]",20
4,6877,"[0.14500641727379412, 0.4097884695072434]",17


['index', 'unit2_pos', 'unit1_beam']


In [7]:
def parse_unit2_pos(pos_value):
    if isinstance(pos_value, str):
        return ast.literal_eval(pos_value)
    return pos_value

def add_position_features(df):
    df = df.copy()

    parsed = df["unit2_pos"].apply(parse_unit2_pos)

    df["pos_x"] = parsed.apply(lambda v: float(v[0]))
    df["pos_y"] = parsed.apply(lambda v: float(v[1]))

    eps = 1e-8

    df["distance"] = np.sqrt(df["pos_x"]**2 + df["pos_y"]**2)
    df["distance2"] = df["distance"] ** 2
    df["distance3"] = df["distance"] ** 3

    df["angle"] = np.arctan2(df["pos_y"], df["pos_x"])
    df["sin_angle"] = np.sin(df["angle"])
    df["cos_angle"] = np.cos(df["angle"])

    df["pos_x2"] = df["pos_x"] ** 2
    df["pos_y2"] = df["pos_y"] ** 2
    df["pos_x3"] = df["pos_x"] ** 3
    df["pos_y3"] = df["pos_y"] ** 3
    df["pos_xy"] = df["pos_x"] * df["pos_y"]

    df["unit_x"] = df["pos_x"] / (df["distance"] + eps)
    df["unit_y"] = df["pos_y"] / (df["distance"] + eps)

    df["sin2_angle"] = np.sin(2 * df["angle"])
    df["cos2_angle"] = np.cos(2 * df["angle"])
    df["sin3_angle"] = np.sin(3 * df["angle"])
    df["cos3_angle"] = np.cos(3 * df["angle"])

    df["dist_sin"] = df["distance"] * df["sin_angle"]
    df["dist_cos"] = df["distance"] * df["cos_angle"]

    return df

train_df_fe = add_position_features(train_df)
val_df_fe   = add_position_features(val_df)
test_df_fe  = add_position_features(test_df)

feature_cols = [
    "pos_x",
    "pos_y",
    "distance",
    "distance2",
    "distance3",
    "angle",
    "sin_angle",
    "cos_angle",
    "pos_x2",
    "pos_y2",
    "pos_x3",
    "pos_y3",
    "pos_xy",
    "unit_x",
    "unit_y",
    "sin2_angle",
    "cos2_angle",
    "sin3_angle",
    "cos3_angle",
    "dist_sin",
    "dist_cos",
]

label_col = "unit1_beam"

print("Number of features:", len(feature_cols))
print("Label column:", label_col)
display(train_df_fe[["index", "unit2_pos", *feature_cols, label_col]].head())

print("Label unique count:", train_df_fe[label_col].nunique())
print("Label min/max:", train_df_fe[label_col].min(), train_df_fe[label_col].max())

Number of features: 21
Label column: unit1_beam


,index,unit2_pos,pos_x,pos_y,distance,distance2,distance3,angle,sin_angle,cos_angle,...,pos_xy,unit_x,unit_y,sin2_angle,cos2_angle,sin3_angle,cos3_angle,dist_sin,dist_cos,unit1_beam
0,3532,"[0.8092883966431671, 0.521083920903955]",0.809288,0.521084,0.962536,0.926476,0.891767,0.572060,0.541365,0.840787,...,0.421707,0.840787,0.541365,0.910347,0.413847,0.989450,-0.144873,0.521084,0.809288,17
1,2224,"[0.4816276084988933, 0.29434536152734486]",0.481628,0.294345,0.564450,0.318604,0.179836,0.548576,0.521472,0.853268,...,0.141765,0.853268,0.521472,0.889912,0.456133,0.997194,-0.074861,0.294345,0.481628,14
2,9416,"[0.220278556834608, 0.4136596156292844]",0.220279,0.413660,0.468654,0.219637,0.102934,1.081479,0.882654,0.470023,...,0.091120,0.470023,0.882654,0.829736,-0.558156,-0.102663,-0.994716,0.413660,0.220279,17
3,8510,"[0.21412273613497904, 0.4547214157104936]",0.214123,0.454721,0.502613,0.252620,0.126970,1.130709,0.904714,0.426019,...,0.097366,0.426019,0.904714,0.770851,-0.637016,-0.247920,-0.968780,0.454721,0.214123,20
4,6877,"[0.14500641727379412, 0.4097884695072434]",0.145006,0.409788,0.434688,0.188953,0.082136,1.230690,0.942719,0.333588,...,0.059422,0.333588,0.942719,0.628959,-0.777439,-0.523094,-0.852275,0.409788,0.145006,17


Label unique count: 29
Label min/max: 2 30


In [8]:
all_labels = sorted(train_df_fe[label_col].astype(int).unique())

label_to_id = {label: i for i, label in enumerate(all_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

num_classes = len(all_labels)

print("num_classes:", num_classes)
print("label_to_id:", label_to_id)
print("id_to_label:", id_to_label)

num_classes: 29
label_to_id: {np.int64(2): 0, np.int64(3): 1, np.int64(4): 2, np.int64(5): 3, np.int64(6): 4, np.int64(7): 5, np.int64(8): 6, np.int64(9): 7, np.int64(10): 8, np.int64(11): 9, np.int64(12): 10, np.int64(13): 11, np.int64(14): 12, np.int64(15): 13, np.int64(16): 14, np.int64(17): 15, np.int64(18): 16, np.int64(19): 17, np.int64(20): 18, np.int64(21): 19, np.int64(22): 20, np.int64(23): 21, np.int64(24): 22, np.int64(25): 23, np.int64(26): 24, np.int64(27): 25, np.int64(28): 26, np.int64(29): 27, np.int64(30): 28}
id_to_label: {0: np.int64(2), 1: np.int64(3), 2: np.int64(4), 3: np.int64(5), 4: np.int64(6), 5: np.int64(7), 6: np.int64(8), 7: np.int64(9), 8: np.int64(10), 9: np.int64(11), 10: np.int64(12), 11: np.int64(13), 12: np.int64(14), 13: np.int64(15), 14: np.int64(16), 15: np.int64(17), 16: np.int64(18), 17: np.int64(19), 18: np.int64(20), 19: np.int64(21), 20: np.int64(22), 21: np.int64(23), 22: np.int64(24), 23: np.int64(25), 24: np.int64(26), 25: np.int64(27), 26

In [9]:
train_mean = train_df_fe[feature_cols].astype(float).mean()
train_std  = train_df_fe[feature_cols].astype(float).std().replace(0, 1)

print("Train mean:")
display(train_mean)

print("Train std:")
display(train_std)

Train mean:


,0
pos_x,0.343933
pos_y,0.396131
distance,0.540083
distance2,0.311916
distance3,0.193269
angle,0.870551
sin_angle,0.744473
cos_angle,0.625777
pos_x2,0.141736
pos_y2,0.170180


Train std:


,0
pos_x,0.153132
pos_y,0.115162
distance,0.142230
distance2,0.177328
distance3,0.181966
angle,0.236080
sin_angle,0.161494
cos_angle,0.167600
pos_x2,0.137104
pos_y2,0.098119


dataset class 

In [10]:
class PositionBeamDataset(Dataset):
    def __init__(self, df, feature_cols, label_col, mean, std, label_to_id):
        self.df = df.reset_index(drop=True).copy()
        self.feature_cols = feature_cols
        self.label_col = label_col
        self.mean = mean
        self.std = std
        self.label_to_id = label_to_id

        x = self.df[self.feature_cols].astype(float)
        x = (x - self.mean) / self.std

        y_raw = self.df[self.label_col].astype(int).values
        y = [self.label_to_id[int(v)] for v in y_raw]

        self.x = torch.tensor(x.values, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [11]:
batch_size = 64

dataset_train = PositionBeamDataset(train_df_fe, feature_cols, label_col, train_mean, train_std, label_to_id)
dataset_val   = PositionBeamDataset(val_df_fe, feature_cols, label_col, train_mean, train_std, label_to_id)
dataset_test  = PositionBeamDataset(test_df_fe, feature_cols, label_col, train_mean, train_std, label_to_id)

train_loader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(dataset_test, batch_size=batch_size, shuffle=False)

print("Train size:", len(dataset_train))
print("Val size  :", len(dataset_val))
print("Test size :", len(dataset_test))

Train size: 6832
Val size  : 3416
Test size : 1139


sanity check 

In [13]:
x_batch, y_batch = next(iter(train_loader))

input_dim = x_batch.shape[1]

print("x_batch:", x_batch.shape)
print("y_batch:", y_batch.shape)
print("input_dim:", input_dim)
print("num_classes:", num_classes)

print("First x:", x_batch[:3])
print("First mapped labels:", y_batch[:10])
print("First original labels:", [id_to_label[int(i)] for i in y_batch[:10]])

assert input_dim == len(feature_cols)
assert y_batch.min().item() >= 0
assert y_batch.max().item() < num_classes

print(" Data pipeline ready.")

x_batch: torch.Size([64, 21])
y_batch: torch.Size([64])
input_dim: 21
num_classes: 29
First x: tensor([[-0.8435,  0.1222, -0.5418, -0.5500, -0.5166,  0.9231,  0.8759, -0.9663,
         -0.6974, -0.0195, -0.5531, -0.1208, -0.5983, -0.9663,  0.8759, -0.4265,
         -0.9630, -1.0518, -0.7390,  0.1222, -0.8435],
        [-0.9343,  0.5208, -0.2933, -0.3583, -0.3819,  1.2090,  1.0571, -1.3290,
         -0.7395,  0.3858, -0.5699,  0.2175, -0.5555, -1.3290,  1.0571, -1.0401,
         -1.2111, -1.4714, -0.6301,  0.5208, -0.9343],
        [ 0.0867,  0.1678,  0.0550, -0.0661, -0.1582, -0.0419,  0.0854,  0.1561,
         -0.1031,  0.0247, -0.2204, -0.0856,  0.1346,  0.1561,  0.0854,  0.7903,
          0.0252,  0.3420, -0.3891,  0.1678,  0.0867]])
First mapped labels: tensor([14, 18, 15, 20, 15, 11, 19, 15, 14, 13])
First original labels: [np.int64(16), np.int64(20), np.int64(17), np.int64(22), np.int64(17), np.int64(13), np.int64(21), np.int64(17), np.int64(16), np.int64(15)]
 Data pipeline read

position mlp model 

In [14]:
class PositionMLP(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dims=(512, 256, 128, 64), dropout=0.10):
        super().__init__()

        h1, h2, h3, h4 = hidden_dims

        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.BatchNorm1d(h1),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(h2, h3),
            nn.BatchNorm1d(h3),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(h3, h4),
            nn.BatchNorm1d(h4),
            nn.GELU(),

            nn.Linear(h4, num_classes)
        )

    def forward(self, x):
        return self.net(x)

top k evaluation 

In [15]:
def evaluate_topk(model, loader, device, ks=(1, 2, 3, 5)):
    model.eval()

    total = 0
    correct = {k: 0 for k in ks}

    with torch.no_grad():
        for x, labels in loader:
            x = x.to(device)
            labels = labels.to(device)

            outputs = model(x)

            max_k = max(ks)
            _, pred = torch.topk(outputs, k=max_k, dim=1)
            pred = pred.t()

            total += labels.size(0)

            for k in ks:
                correct[k] += pred[:k].eq(labels.view(1, -1)).sum().item()

    return {f"top{k}": 100.0 * correct[k] / total for k in ks}

trainer 

In [17]:
def train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=120,
    lr=2e-4,
    weight_decay=1e-6,
    milestones=(70, 100),
    save_path="/content/drive/MyDrive/best_position_mlp_tracking.pth"
):
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=list(milestones),
        gamma=0.1
    )

    best_top1 = -1
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        total_train = 0

        for x, labels in train_loader:
            x = x.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            bs = labels.size(0)
            running_loss += loss.item() * bs
            total_train += bs

        scheduler.step()

        train_loss = running_loss / total_train
        val_metrics = evaluate_topk(model, val_loader, device, ks=(1, 2, 3, 5))

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            **val_metrics
        })

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {train_loss:.4f} | "
            f"Val Top1: {val_metrics['top1']:.2f} | "
            f"Top2: {val_metrics['top2']:.2f} | "
            f"Top3: {val_metrics['top3']:.2f} | "
            f"Top5: {val_metrics['top5']:.2f}"
        )

        if val_metrics["top1"] > best_top1:
            best_top1 = val_metrics["top1"]
            torch.save(copy.deepcopy(model.state_dict()), save_path)
            print("Saved best model")

    return pd.DataFrame(history)

train position mlp 

In [18]:
set_seed(42)

model = PositionMLP(
    input_dim=input_dim,
    num_classes=num_classes,
    hidden_dims=(512, 256, 128, 64),
    dropout=0.10
).to(device)

save_path = "/content/drive/MyDrive/best_position_mlp_tracking.pth"

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=120,
    lr=2e-4,
    weight_decay=1e-6,
    milestones=(70, 100),
    save_path=save_path
)

Epoch 001 | Loss: 2.7303 | Val Top1: 48.77 | Top2: 68.00 | Top3: 77.75 | Top5: 86.45
Saved best model
Epoch 002 | Loss: 2.1074 | Val Top1: 50.53 | Top2: 72.25 | Top3: 83.02 | Top5: 92.56
Saved best model
Epoch 003 | Loss: 1.8073 | Val Top1: 51.49 | Top2: 74.00 | Top3: 85.28 | Top5: 93.79
Saved best model
Epoch 004 | Loss: 1.6225 | Val Top1: 55.09 | Top2: 77.08 | Top3: 87.18 | Top5: 94.09
Saved best model
Epoch 005 | Loss: 1.5295 | Val Top1: 54.45 | Top2: 77.49 | Top3: 87.56 | Top5: 94.85
Epoch 006 | Loss: 1.4526 | Val Top1: 54.45 | Top2: 77.20 | Top3: 87.68 | Top5: 94.99
Epoch 007 | Loss: 1.4200 | Val Top1: 56.59 | Top2: 78.78 | Top3: 88.00 | Top5: 95.35
Saved best model
Epoch 008 | Loss: 1.3785 | Val Top1: 55.42 | Top2: 78.45 | Top3: 88.38 | Top5: 95.02
Epoch 009 | Loss: 1.3724 | Val Top1: 55.91 | Top2: 78.19 | Top3: 87.70 | Top5: 95.20
Epoch 010 | Loss: 1.3377 | Val Top1: 57.20 | Top2: 78.92 | Top3: 88.11 | Top5: 95.43
Saved best model
Epoch 011 | Loss: 1.3275 | Val Top1: 58.40 | Top

In [ ]:
model_eval = PositionMLP(
    input_dim=input_dim,
    num_classes=num_classes,
    hidden_dims=(512, 256, 128, 64),
    dropout=0.10
).to(device)

model_eval.load_state_dict(torch.load(save_path, map_location=device))

test_metrics = evaluate_topk(model_eval, test_loader, device, ks=(1, 2, 3, 5))

print("Position MLP Test Metrics:")
print(test_metrics)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["epoch"], history["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Train Loss")
plt.title("Position MLP Training Loss")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history["epoch"], history["top1"], label="Top-1")
plt.plot(history["epoch"], history["top2"], label="Top-2")
plt.plot(history["epoch"], history["top3"], label="Top-3")
plt.plot(history["epoch"], history["top5"], label="Top-5")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.title("Position MLP Validation Top-K")
plt.legend()
plt.grid(True)
plt.show()